In [65]:
# Filename: train_model_advanced_ensemble.py

# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error # Removed make_scorer, r2_score if not used
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb
import geopy.distance
import os
import time
import matplotlib.pyplot as plt
import traceback
import joblib
import warnings
import sys

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# --- Configuration ---
# Data and Paths
TRAINING_RAW_DATA_FILE = 'dataset/merged_5G_DL_noTime_noBlank.csv'
RESULTS_DIR = 'result_train'
MODEL_SAVE_DIR = 'saved_models'
TRAINING_PREFIX = "DL_Ensemble_AdvFeat_v1" # Unique prefix for this model version

# Model Choice and Training Behavior
MODEL_TYPE = 'ENSEMBLE' # Options: 'ENSEMBLE', 'LGBM', 'RF', 'KNN', 'MLP', 'SVR', 'XGB'
USE_GRID_SEARCH = False # For ENSEMBLE, this is False. For individual models, set to True for tuning.
REPROCESS_TRAINING_DATA = True # Set to False if processed data exists and is up-to-date

# Feature Engineering Config (MUST BE IDENTICAL IN predict_locations_kalman.py)
UNIQUE_PCIS = sorted([3, 13, 23, 30, 40, 48, 59, 68, 76])
FINGERPRINT_COLS_RSRP = [f'RSRP_PCI{pci}' for pci in UNIQUE_PCIS]
FINGERPRINT_COLS_RSRQ = [f'RSRQ_PCI{pci}' for pci in UNIQUE_PCIS]
FINGERPRINT_COLS_OTHER = ['SINR_Serving', 'TA_Serving']
BASE_FINGERPRINT_FEATURES = FINGERPRINT_COLS_RSRP + FINGERPRINT_COLS_RSRQ + FINGERPRINT_COLS_OTHER
NEW_NUMERICAL_RAW_COLS = [
    'merged_row_count', 'NR_UE_Pathloss_DL_0', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL',
    'NR_UE_NACK_Rate_DL_0', 'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_MCS_DL_0', 'NR_UE_RB_Num_DL_0',
    'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_BLER_DL_0', 'NR_UE_CCE_AggregationLev_0',
    'NR_UE_Power_Tx_PUSCH_0', 'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0',
    'NR_UE_RACH_Attempt', 'NR_UE_RACH_OK', 'NR_UE_RACH_Fail', 'NR_UE_RACH_Procedure_Count',
    'NR_UE_RRCReEstAttempt', 'NR_UE_RRCReEstFail', 'NR_UE_RRCConnectionAttempt',
    'NR_UE_RRCConnectionSetupOk', 'NR_UE_RRCConnectionComplete', 'NR_UE_RRCConnectionDrop',
    'NR_UE_RRCHOAttempt', 'NR_UE_RRCHOOK'
]
CATEGORICAL_RAW_COLS_CONFIG = { # Renamed from CATEGORICAL_RAW_COLS for clarity
    'Technology_Mode': {'default': 'Unknown_TechMode'},
    'NR_UE_RRCReEst_EndResult': {'default': 'Unknown_ReEstResult'}
}
LABEL_COLS = ['Latitude', 'Longitude']

# Default values for fingerprinting and new features
NO_SIGNAL_RSRP_DBM = -140.0; NO_SIGNAL_RSRQ_DB = -30.0
DEFAULT_SINR_DB = -20.0; DEFAULT_TA = 0.0
DEFAULT_VALUES_NEW_FEATURES = {
    'merged_row_count': 1.0, 'NR_UE_Pathloss_DL_0': 150.0, 'NR_UE_Throughput_PDCP_DL': 0.0,
    'App_Throughput_DL': 0.0, 'NR_UE_NACK_Rate_DL_0': 0.0, 'NR_UE_Ack_As_Nack_DL_0': 0.0,
    'NR_UE_MCS_DL_0': 0.0, 'NR_UE_RB_Num_DL_0': 0.0, 'NR_UE_Modulation_Avg_DL_0': 0.0,
    'NR_UE_RI_DL_0': 1.0, 'NR_UE_BLER_DL_0': 0.0, 'NR_UE_CCE_AggregationLev_0': 4.0,
    'NR_UE_Power_Tx_PUSCH_0': 0.0, 'NR_UE_Power_Tx_PRACH_0': 0.0, 'NR_UE_NACK_Rate_UL_0': 0.0,
    'NR_UE_RACH_Attempt': 0.0, 'NR_UE_RACH_OK': 0.0, 'NR_UE_RACH_Fail': 0.0,
    'NR_UE_RACH_Procedure_Count': 0.0, 'NR_UE_RRCReEstAttempt': 0.0, 'NR_UE_RRCReEstFail': 0.0,
    'NR_UE_RRCConnectionAttempt': 0.0, 'NR_UE_RRCConnectionSetupOk': 0.0,
    'NR_UE_RRCConnectionComplete': 0.0, 'NR_UE_RRCConnectionDrop': 0.0,
    'NR_UE_RRCHOAttempt': 0.0, 'NR_UE_RRCHOOK': 0.0
}
ALL_RAW_FEATURE_COL_NAMES = BASE_FINGERPRINT_FEATURES + NEW_NUMERICAL_RAW_COLS + list(CATEGORICAL_RAW_COLS_CONFIG.keys())


# Model Hyperparameters
# For Ensemble members (use pre-tuned or good defaults)
LGBM_ENSEMBLE_PARAMS = {'objective': 'regression_l2', 'n_estimators': 200, 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': -1, 'random_state': 42, 'n_jobs': 1, 'verbose':-1}
RF_ENSEMBLE_PARAMS = {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42, 'n_jobs': 1}
KNN_ENSEMBLE_PARAMS = {'n_neighbors': 7, 'weights': 'distance', 'metric': 'manhattan', 'n_jobs': 1}

# Full Grids (if USE_GRID_SEARCH is True for individual models)
# ... (Keep your DEFAULT_*_PARAMS and *_GRIDSEARCH_PARAM_GRID definitions here as before) ...
DEFAULT_KNN_K_NEIGHBORS = 12; DEFAULT_KNN_WEIGHTS = 'distance'; DEFAULT_KNN_METRIC = 'manhattan'
KNN_GRIDSEARCH_PARAM_GRID = {'n_neighbors': list(range(3, 16, 2)), 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}
DEFAULT_LGBM_PARAMS = {'objective': 'regression_l2', 'metric': 'rmse', 'n_estimators': 376, 'learning_rate': 0.052, 'num_leaves': 31, 'max_depth': -1, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'verbose': -1, 'n_jobs': -1, 'seed': 42}
LGBM_GRIDSEARCH_PARAM_GRID = {'estimator__n_estimators': [200, 500, 1000], 'estimator__learning_rate': [0.02, 0.05, 0.1], 'estimator__num_leaves': [20, 31, 45], 'estimator__max_depth': [-1, 10, 20]}
DEFAULT_MLP_PARAMS = {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.01, 'batch_size': 'auto', 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 300, 'shuffle': True, 'random_state': 42, 'tol': 1e-4, 'verbose': False, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10}
MLP_GRIDSEARCH_PARAM_GRID = {'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)], 'activation': ['relu', 'tanh'], 'alpha': [0.0001, 0.001, 0.01], 'learning_rate_init': [0.001, 0.0005]}
DEFAULT_SVR_PARAMS = {'C': 1.0, 'epsilon': 0.1, 'kernel': 'rbf', 'gamma': 'scale'}
SVR_GRIDSEARCH_PARAM_GRID = {'estimator__kernel': ['rbf', 'linear'], 'estimator__C': [0.1, 1, 10, 50], 'estimator__gamma': ['scale', 'auto'], 'estimator__epsilon': [0.01, 0.1, 0.2]}
DEFAULT_RF_PARAMS = {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'random_state': 42, 'n_jobs': -1}
RF_GRIDSEARCH_PARAM_GRID = {'n_estimators': [100, 200, 300], 'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
DEFAULT_XGB_PARAMS = {'objective': 'reg:squarederror', 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1}
XGB_GRIDSEARCH_PARAM_GRID = {'estimator__n_estimators': [100, 200, 500], 'estimator__learning_rate': [0.05, 0.1, 0.2], 'estimator__max_depth': [3, 5, 7], 'estimator__subsample': [0.7, 0.8, 0.9], 'estimator__colsample_bytree': [0.7, 0.8, 0.9]}


GRIDSEARCH_CV_FOLDS = 5
GRIDSEARCH_N_JOBS = -1
USE_SCALING_IN_COLUMNTRANSFORMER = True # Controls if StandardScaler is used

# --- File Paths ---
PROCESSED_TRAINING_DATA_FILE = os.path.join(RESULTS_DIR, f'{TRAINING_PREFIX}_processed_fingerprints.csv')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# --- Functions ---
def process_raw_data_train(df_raw_input, label_cols_list, all_raw_feature_cols_list):
    """Processes raw data to create the feature set before transformation."""
    print("Processing raw training data (to create feature set for ColumnTransformer)...")
    start_time = time.time()
    
    missing_labels = [col for col in label_cols_list if col not in df_raw_input.columns]
    if missing_labels:
        print(f"FATAL: Label columns missing in raw data: {missing_labels}"); return None
    
    df_processed = df_raw_input.dropna(subset=label_cols_list).copy()
    if len(df_processed) < len(df_raw_input):
        print(f"Dropped {len(df_raw_input) - len(df_processed)} rows with missing Lat/Lon.")
    if df_processed.empty:
        print("FATAL: No valid rows after dropping missing Lat/Lon."); return None

    # 1. Initialize and Populate Base Fingerprint Features
    for pci in UNIQUE_PCIS: # Global UNIQUE_PCIS
        df_processed[f'RSRP_PCI{pci}'] = NO_SIGNAL_RSRP_DBM
        df_processed[f'RSRQ_PCI{pci}'] = NO_SIGNAL_RSRQ_DB
    df_processed['SINR_Serving'] = DEFAULT_SINR_DB
    df_processed['TA_Serving'] = DEFAULT_TA

    # Names of columns in the *input* df_raw_input for serving/neighbor info
    # These might differ from the predict script if CSV headers are different
    srv_pci_col, srv_rsrp_col, srv_rsrq_col = 'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0'
    srv_sinr_col, srv_ta_col = 'NR_UE_SINR_0', 'NR_UE_Timing_Advance'
    nbr_pci_pat, nbr_rsrp_pat, nbr_rsrq_pat = 'NR_UE_Nbr_PCI_{}', 'NR_UE_Nbr_RSRP_{}', 'NR_UE_Nbr_RSRQ_{}'
    max_nbrs = 5

    for index, row in df_processed.iterrows():
        s_pci = row.get(srv_pci_col)
        if pd.notna(s_pci) and int(s_pci) in UNIQUE_PCIS:
            s_pci_int = int(s_pci)
            if pd.notna(row.get(srv_rsrp_col)): df_processed.loc[index, f'RSRP_PCI{s_pci_int}'] = float(row[srv_rsrp_col])
            if pd.notna(row.get(srv_rsrq_col)): df_processed.loc[index, f'RSRQ_PCI{s_pci_int}'] = float(row[srv_rsrq_col])
        if pd.notna(row.get(srv_sinr_col)): df_processed.loc[index, 'SINR_Serving'] = float(row[srv_sinr_col])
        if pd.notna(row.get(srv_ta_col)): df_processed.loc[index, 'TA_Serving'] = float(row[srv_ta_col])
        for i in range(max_nbrs):
            nbr_pci_val = row.get(nbr_pci_pat.format(i))
            if pd.notna(nbr_pci_val) and int(nbr_pci_val) in UNIQUE_PCIS:
                nbr_pci_int = int(nbr_pci_val)
                if pd.notna(row.get(nbr_rsrp_pat.format(i))): df_processed.loc[index, f'RSRP_PCI{nbr_pci_int}'] = float(row[nbr_rsrp_pat.format(i)])
                if pd.notna(row.get(nbr_rsrq_pat.format(i))): df_processed.loc[index, f'RSRQ_PCI{nbr_pci_int}'] = float(row[nbr_rsrq_pat.format(i)])

    # 2. Handle New Numerical Features
    for col in NEW_NUMERICAL_RAW_COLS: # Global NEW_NUMERICAL_RAW_COLS
        if col in df_processed.columns:
            df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
            df_processed[col].fillna(DEFAULT_VALUES_NEW_FEATURES[col], inplace=True)
        else:
            print(f"Warning: Numerical column '{col}' not found in training data. Creating with default.")
            df_processed[col] = DEFAULT_VALUES_NEW_FEATURES[col]
            
    # 3. Handle Categorical Features
    for col, params in CATEGORICAL_RAW_COLS_CONFIG.items(): # Global CATEGORICAL_RAW_COLS_CONFIG
        if col in df_processed.columns:
            df_processed[col].fillna(params['default'], inplace=True)
        else:
            print(f"Warning: Categorical column '{col}' not found in training data. Creating with default.")
            df_processed[col] = params['default']
            
    # Select only the label columns and the defined raw feature columns
    final_columns_to_keep = label_cols_list + all_raw_feature_cols_list
    
    # Ensure all expected raw feature columns exist in df_processed before selection
    for col in all_raw_feature_cols_list:
        if col not in df_processed.columns:
            print(f"CRITICAL Error during final column assembly: Feature '{col}' is missing from processed df.")
            # This implies an issue in earlier steps or config if a defined feature isn't created
            # For robustness, one might create it here with a default, but it's better to fix the source
            # df_processed[col] = 0 # Example emergency fallback
            return None 
            
    df_final_output = df_processed[final_columns_to_keep].copy()
    
    elapsed_time = time.time() - start_time
    print(f"Raw feature processing finished in {elapsed_time:.2f}s. Shape: {df_final_output.shape}")
    return df_final_output

def calculate_errors(y_true_df, y_pred_array, model_name_for_print="Model"):
    errors, error_details, invalid_preds = [], [], 0
    if not isinstance(y_true_df, pd.DataFrame) or not all(col in y_true_df.columns for col in ['Latitude', 'Longitude']): return np.array([]), pd.DataFrame()
    if not isinstance(y_pred_array, np.ndarray) or y_pred_array.shape[1] != 2: return np.array([]), pd.DataFrame()
    if len(y_true_df) != len(y_pred_array): return np.array([]), pd.DataFrame()
    y_true_df_reset = y_true_df.reset_index(drop=True)
    lat_true_col, lon_true_col = y_true_df_reset['Latitude'].values, y_true_df_reset['Longitude'].values
    num_skipped_val_error = 0
    for i in range(len(y_true_df_reset)):
        true_lat, true_lon, pred_lat, pred_lon = lat_true_col[i], lon_true_col[i], y_pred_array[i,0], y_pred_array[i,1]
        current_error_detail = {'true_lat': true_lat, 'true_lon': true_lon, 'pred_lat': pred_lat, 'pred_lon': pred_lon, 'error_m': np.nan}
        if not all(pd.notna(val) for val in [true_lat, true_lon, pred_lat, pred_lon]): 
            invalid_preds += 1; error_details.append(current_error_detail); continue
        if not (-90 <= pred_lat <= 90 and -180 <= pred_lon <= 180):
            invalid_preds +=1; num_skipped_val_error +=1; error_details.append(current_error_detail); continue
        try:
            dist = geopy.distance.geodesic((true_lat, true_lon), (pred_lat, pred_lon)).meters
            errors.append(dist); current_error_detail['error_m'] = dist; error_details.append(current_error_detail)
        except ValueError: invalid_preds += 1; num_skipped_val_error +=1; error_details.append(current_error_detail)
    print(f"({model_name_for_print}) Calculated errors for {len(errors)} points. Invalid/skipped: {invalid_preds}/{len(y_true_df_reset)} (Val errors: {num_skipped_val_error}).")
    return np.array(errors), pd.DataFrame(error_details)

def format_params_for_filename(params_input):
    if not params_input: return "default"
    if isinstance(params_input, str): return params_input.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
    items = []
    for k, v in sorted(params_input.items()):
        key_name = k.split('__')[-1]
        if isinstance(v, tuple): val_str = "x".join(map(str, v))
        elif isinstance(v, float): val_str = f"{v:.0e}"
        else: val_str = str(v)
        items.append(f"{key_name}_{val_str}")
    full_str = "_".join(items).replace("'", "").replace(" ", "").replace(",", "_").replace(".", "p")
    return full_str[:100] # Increased max length slightly for descriptive ensemble names

# --- Main Training Script ---
if __name__ == '__main__':
    print(f"--- Starting Training Script: {TRAINING_PREFIX} ---")
    script_start_time = time.time()

    # 1. Load and Process Training Data
    print(f"\n[Step 1/7] Loading/Processing Raw Training Data...")
    processed_df = None
    column_transformer_fitted = None

    if REPROCESS_TRAINING_DATA or not os.path.exists(PROCESSED_TRAINING_DATA_FILE):
        print(f"Mode: Reprocessing raw data from {TRAINING_RAW_DATA_FILE}...")
        try:
            raw_df = pd.read_csv(TRAINING_RAW_DATA_FILE, low_memory=False)
            print(f"Raw training data loaded: {len(raw_df)} rows.")
            processed_df = process_raw_data_train(raw_df, LABEL_COLS, ALL_RAW_FEATURE_COL_NAMES)
            if processed_df is None or processed_df.empty:
                raise ValueError("Data processing returned empty or None DataFrame.")
            processed_df.to_csv(PROCESSED_TRAINING_DATA_FILE, index=False)
            print(f"Processed training data saved to {PROCESSED_TRAINING_DATA_FILE}")
        except FileNotFoundError:
            print(f"FATAL: Training raw data file not found: {TRAINING_RAW_DATA_FILE}"); sys.exit()
        except Exception as e:
            print(f"FATAL: Error during data loading/processing: {e}"); traceback.print_exc(); sys.exit()
    else:
        print(f"Mode: Loading existing pre-processed data from {PROCESSED_TRAINING_DATA_FILE}...")
        try:
            processed_df = pd.read_csv(PROCESSED_TRAINING_DATA_FILE)
            print(f"Pre-processed data loaded: {len(processed_df)} samples.")
            if processed_df.empty:
                raise ValueError("Loaded pre-processed file is empty.")
        except Exception as e:
            print(f"FATAL: Error loading pre-processed data: {e}"); traceback.print_exc(); sys.exit()

    if processed_df is None: print("FATAL: processed_df is None."); sys.exit()

    # 2. Prepare Features (X) and Labels (y)
    print(f"\n[Step 2/7] Preparing Features (X) and Labels (y)...")
    try:
        X_raw = processed_df[ALL_RAW_FEATURE_COL_NAMES] # These are features before transformation
        y = processed_df[LABEL_COLS]
        print(f"Raw X features shape: {X_raw.shape}, y labels shape: {y.shape}")
    except KeyError as e:
        print(f"FATAL: KeyError preparing X,y. Missing columns: {e}. Available: {processed_df.columns.tolist()}"); sys.exit()
    except Exception as e:
        print(f"FATAL: Error preparing X,y: {e}"); traceback.print_exc(); sys.exit()

    # 3. Split Data
    print(f"\n[Step 3/7] Splitting data into Training/Validation sets (80/20)...")
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_raw, y, test_size=0.2, random_state=42)
    print(f"Raw features: Train samples: {len(X_train_raw)}, Validation samples: {len(X_val_raw)}")

    # 4. Feature Preprocessing (ColumnTransformer)
    print(f"\n[Step 4/7] Feature Preprocessing (Scaling/OHE)...")
    numerical_cols_ct = [col for col in ALL_RAW_FEATURE_COL_NAMES if col in (BASE_FINGERPRINT_FEATURES + NEW_NUMERICAL_RAW_COLS)]
    categorical_cols_ct = [col for col in ALL_RAW_FEATURE_COL_NAMES if col in CATEGORICAL_RAW_COLS_CONFIG.keys()]
    
    transformers_ct = []
    if numerical_cols_ct:
        if USE_SCALING_IN_COLUMNTRANSFORMER:
            transformers_ct.append(('num', StandardScaler(), numerical_cols_ct))
            print(f"  Applying StandardScaler to {len(numerical_cols_ct)} numerical columns.")
        else:
            transformers_ct.append(('num_pass', 'passthrough', numerical_cols_ct))
            print(f"  Passing through {len(numerical_cols_ct)} numerical columns (no scaling).")
            
    if categorical_cols_ct:
        transformers_ct.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_ct))
        print(f"  Applying OneHotEncoder to: {categorical_cols_ct}")

    if not transformers_ct:
        print("Warning: No transformers defined (e.g. no numerical cols for scaling, no categorical for OHE). Using raw features as is.")
        X_train_processed = X_train_raw.values if not X_train_raw.empty else np.array([])
        X_val_processed = X_val_raw.values if not X_val_raw.empty else np.array([])
        column_transformer_fitted = None
    else:
        column_transformer_fitted = ColumnTransformer(transformers_ct, remainder='drop') # Drop any cols not specified
        try:
            X_train_processed = column_transformer_fitted.fit_transform(X_train_raw)
            X_val_processed = column_transformer_fitted.transform(X_val_raw)
            print(f"Feature preprocessing complete. Processed shapes: Train {X_train_processed.shape}, Val {X_val_processed.shape}")
        except Exception as e:
            print(f"FATAL: Error during ColumnTransformer fit/transform: {e}"); traceback.print_exc(); sys.exit()
    
    # 5. Model Training / Tuning
    print(f"\n[Step 5/7] Training/Tuning Model ({MODEL_TYPE})...")
    current_model = None; best_hyperparams = {}; model_display_name = MODEL_TYPE.upper(); training_duration = 0
    
    try:
        if MODEL_TYPE == 'ENSEMBLE':
            model_display_name = "Ensemble_MOR_Voting_LGBM_RF_KNN"
            print(f"--- Training {model_display_name} (MultiOutputRegressor wrapping VotingRegressor) ---")
            lgbm_st = lgb.LGBMRegressor(**LGBM_ENSEMBLE_PARAMS)
            rf_st = RandomForestRegressor(**RF_ENSEMBLE_PARAMS)
            knn_st = KNeighborsRegressor(**KNN_ENSEMBLE_PARAMS)
            voting_reg_st = VotingRegressor(
                estimators=[('lgbm', lgbm_st), ('rf', rf_st), ('knn', knn_st)],
                n_jobs=-1 
            )
            current_model = MultiOutputRegressor(voting_reg_st, n_jobs=1) # n_jobs for MOR to fit 2 voters
            best_hyperparams = model_display_name # For file naming
            training_start_time = time.time()
            current_model.fit(X_train_processed, y_train)
            training_duration = time.time() - training_start_time
            print(f"Ensemble model training finished in {training_duration:.2f} seconds.")
        else: # Individual model training logic
            # (This block is the same as your previous individual model training logic)
            # Select base_model, param_grid, default_params based on MODEL_TYPE
            base_model_instance = None
            param_grid_dict = {}
            default_params_dict = {}

            if MODEL_TYPE == 'MLP':
                base_model_instance = MLPRegressor(random_state=42, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10)
                param_grid_dict = MLP_GRIDSEARCH_PARAM_GRID; default_params_dict = DEFAULT_MLP_PARAMS
            elif MODEL_TYPE == 'LGBM':
                base_lgbm = lgb.LGBMRegressor(objective=DEFAULT_LGBM_PARAMS.get('objective', 'regression_l2'), metric=DEFAULT_LGBM_PARAMS.get('metric', 'rmse'), verbose=DEFAULT_LGBM_PARAMS.get('verbose', -1), n_jobs=1, random_state=DEFAULT_LGBM_PARAMS.get('seed', 42))
                base_model_instance = MultiOutputRegressor(base_lgbm, n_jobs=None); param_grid_dict = LGBM_GRIDSEARCH_PARAM_GRID; default_params_dict = DEFAULT_LGBM_PARAMS
            elif MODEL_TYPE == 'KNN':
                base_model_instance = KNeighborsRegressor(n_jobs=1); param_grid_dict = KNN_GRIDSEARCH_PARAM_GRID
                default_params_dict = {'n_neighbors': DEFAULT_KNN_K_NEIGHBORS, 'weights': DEFAULT_KNN_WEIGHTS, 'metric': DEFAULT_KNN_METRIC, 'n_jobs': -1}
            elif MODEL_TYPE == 'SVR':
                base_svr_instance = SVR(); base_model_instance = MultiOutputRegressor(base_svr_instance); param_grid_dict = SVR_GRIDSEARCH_PARAM_GRID; default_params_dict = DEFAULT_SVR_PARAMS
            elif MODEL_TYPE == 'RF':
                base_model_instance = RandomForestRegressor(random_state=42, n_jobs=1); param_grid_dict = RF_GRIDSEARCH_PARAM_GRID; default_params_dict = DEFAULT_RF_PARAMS
            elif MODEL_TYPE == 'XGB':
                base_xgb_instance = xgb.XGBRegressor(objective=DEFAULT_XGB_PARAMS.get('objective','reg:squarederror'), random_state=DEFAULT_XGB_PARAMS.get('random_state', 42), n_jobs=1)
                base_model_instance = MultiOutputRegressor(base_xgb_instance, n_jobs=None); param_grid_dict = XGB_GRIDSEARCH_PARAM_GRID; default_params_dict = DEFAULT_XGB_PARAMS
            else:
                raise ValueError(f"FATAL: Unknown MODEL_TYPE '{MODEL_TYPE}' for individual training.")

            if USE_GRID_SEARCH:
                print(f"--- Hyperparameter Tuning for {model_display_name} with GridSearchCV ---")
                scoring_metric_gs = 'r2' if MODEL_TYPE == 'SVR' else 'neg_mean_squared_error'
                grid_search_cv = GridSearchCV(estimator=base_model_instance, param_grid=param_grid_dict, cv=GRIDSEARCH_CV_FOLDS,
                                        scoring=scoring_metric_gs, refit=True, verbose=1, n_jobs=GRIDSEARCH_N_JOBS)
                training_start_time = time.time()
                grid_search_cv.fit(X_train_processed, y_train)
                training_duration = time.time() - training_start_time
                current_model = grid_search_cv.best_estimator_
                best_hyperparams = grid_search_cv.best_params_
                print(f"GridSearchCV finished in {training_duration:.2f}s. Best params: {best_hyperparams}. Best CV score: {grid_search_cv.best_score_:.4f}")
            else:
                print(f"--- Training {model_display_name} with Predefined Parameters ---")
                best_hyperparams = default_params_dict
                if isinstance(base_model_instance, MultiOutputRegressor):
                    inner_params = {k.split('__')[-1]: v for k, v in default_params_dict.items() if k.split('__')[-1] in base_model_instance.estimator.get_params()}
                    if 'n_jobs' in default_params_dict and 'n_jobs' not in inner_params and hasattr(base_model_instance, 'n_jobs'): base_model_instance.n_jobs = default_params_dict['n_jobs']
                    base_model_instance.estimator.set_params(**inner_params)
                else:
                    valid_params = {k: v for k,v in default_params_dict.items() if k in base_model_instance.get_params()}
                    base_model_instance.set_params(**valid_params)
                current_model = base_model_instance
                training_start_time = time.time()
                current_model.fit(X_train_processed, y_train)
                training_duration = time.time() - training_start_time
                print(f"Manual {model_display_name} training finished in {training_duration:.2f}s.")

        if current_model is None: raise RuntimeError("Model not trained.")
    except Exception as e:
        print(f"FATAL: Error during model training/tuning: {e}"); traceback.print_exc(); sys.exit()
    print(f"--- {model_display_name} Model Trained ---")

    # File naming and saving paths
    filename_params_suffix = format_params_for_filename(best_hyperparams)
    base_save_filename = f'{TRAINING_PREFIX}_{filename_params_suffix}'
    SAVED_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, f'{base_save_filename}_model.joblib')
    SAVED_TRANSFORMER_PATH = os.path.join(MODEL_SAVE_DIR, f'{base_save_filename}_columntransformer.joblib')
    
    # 6. Evaluate Model
    print(f"\n[Step 6/7] Evaluating trained {model_display_name} model on validation set...")
    eval_start_time = time.time(); val_errors_list = np.array([]); val_error_details_df = pd.DataFrame()
    try:
        y_pred_validation = current_model.predict(X_val_processed)
        val_errors_list, val_error_details_df = calculate_errors(y_val, y_pred_validation, model_display_name)
    except Exception as e: print(f"Error during validation: {e}"); traceback.print_exc()
    eval_duration = time.time() - eval_start_time
    
    VAL_ERROR_CSV_PATH = os.path.join(RESULTS_DIR, f'{base_save_filename}_validation_errors.csv')
    VAL_PLOT_PNG_PATH = VAL_ERROR_CSV_PATH.replace('.csv', '_cdf.png')

    if not val_error_details_df.empty:
        try: val_error_details_df.to_csv(VAL_ERROR_CSV_PATH, index=False, float_format='%.6f'); print(f"Saved validation errors to {VAL_ERROR_CSV_PATH}")
        except Exception as e: print(f"Error saving validation errors: {e}")

    if len(val_errors_list) > 0:
        mean_val_err = np.mean(val_errors_list); median_val_err = np.median(val_errors_list)
        rmse_val_dist = np.sqrt(np.mean(np.square(val_errors_list)))
        print(f"\n--- Validation Performance ({model_display_name}) ---")
        params_report = filename_params_suffix if MODEL_TYPE == 'ENSEMBLE' else (", ".join(f"{k.split('__')[-1]}={v}" for k,v in best_hyperparams.items()) if isinstance(best_hyperparams, dict) else str(best_hyperparams))
        print(f"Params: {params_report}\nValid points: {len(val_errors_list)}\n")
        print(f"Mean Error:   {mean_val_err:.2f} m\nMedian Error: {median_val_err:.2f} m\nRMSE (dist):  {rmse_val_dist:.2f} m")
        percentiles_calc = [50,68,80,90,95]; percentile_vals = np.percentile(val_errors_list, percentiles_calc)
        print("Percentiles:", ", ".join([f"{p}%={v:.1f}m" for p,v in zip(percentiles_calc, percentile_vals)]),"\n---------------------------------------\n")
        try:
            plt.figure(figsize=(8,6)); sorted_val_errors=np.sort(val_errors_list); cdf_data=np.arange(1,len(sorted_val_errors)+1)/len(sorted_val_errors)
            plt.plot(sorted_val_errors, cdf_data*100, marker='.', linestyle='-', markersize=2)
            plt.xlabel("Positioning Error (m)"); plt.ylabel("CDF (%)"); plt.title(f"Validation Error CDF - {model_display_name}\n({params_report})",fontsize=10)
            plt.grid(True,which='both',linestyle='--',linewidth=0.5); plt.ylim(0,100); plt.xlim(left=0)
            plt.axvline(median_val_err,color='orange',linestyle=':',lw=1,label=f'Median ({median_val_err:.1f}m)')
            p95_val_plot=np.percentile(val_errors_list,95); plt.axvline(p95_val_plot,color='red',linestyle=':',lw=1,label=f'95th ({p95_val_plot:.1f}m)')
            plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(VAL_PLOT_PNG_PATH); print(f"Saved CDF plot to {VAL_PLOT_PNG_PATH}"); plt.close()
        except Exception as e: print(f"Error plotting CDF: {e}")
    else: print("No valid errors for validation set.\n")
    print(f"Evaluation finished in {eval_duration:.2f} seconds.")

    # 7. Save Model and ColumnTransformer
    print(f"\n[Step 7/7] Saving Trained Model and ColumnTransformer...")
    save_start_time = time.time()
    try:
        joblib.dump(current_model, SAVED_MODEL_PATH); print(f"Model saved to: {SAVED_MODEL_PATH}")
        if column_transformer_fitted is not None:
            joblib.dump(column_transformer_fitted, SAVED_TRANSFORMER_PATH); print(f"Transformer saved to: {SAVED_TRANSFORMER_PATH}")
        else: print("No ColumnTransformer was fitted/used, not saving one.")
    except Exception as e: print(f"FATAL: Error saving model/transformer: {e}"); traceback.print_exc()
    save_duration = time.time() - save_start_time; print(f"Saving finished in {save_duration:.2f} seconds.")

    script_total_time = time.time() - script_start_time
    print(f"\n--- Training Script Finished in {script_total_time:.2f} seconds ({script_total_time/60:.1f} minutes) ---")
    print("\nRecommendations:\n- Review validation performance. If errors are high, revisit feature engineering or model choices.")
    print("- For ensemble, ensure member models are individually well-tuned for best results.")
    print("- Consider StackingRegressor for potentially higher accuracy if VotingRegressor isn't sufficient.")
    print(f"- To use this model, update 'predict_locations_kalman.py' with these filenames:\n  Model: {SAVED_MODEL_PATH}\n  Transformer: {SAVED_TRANSFORMER_PATH}")

--- Starting Training Script: DL_Ensemble_AdvFeat_v1 ---

[Step 1/7] Loading/Processing Raw Training Data...
Mode: Reprocessing raw data from dataset/merged_5G_DL_noTime_noBlank.csv...
Raw training data loaded: 1117 rows.
Processing raw training data (to create feature set for ColumnTransformer)...
Raw feature processing finished in 1.16s. Shape: (1117, 51)
Processed training data saved to result_train\DL_Ensemble_AdvFeat_v1_processed_fingerprints.csv

[Step 2/7] Preparing Features (X) and Labels (y)...
Raw X features shape: (1117, 49), y labels shape: (1117, 2)

[Step 3/7] Splitting data into Training/Validation sets (80/20)...
Raw features: Train samples: 893, Validation samples: 224

[Step 4/7] Feature Preprocessing (Scaling/OHE)...
  Applying StandardScaler to 47 numerical columns.
  Applying OneHotEncoder to: ['Technology_Mode', 'NR_UE_RRCReEst_EndResult']
Feature preprocessing complete. Processed shapes: Train (893, 50), Val (224, 50)

[Step 5/7] Training/Tuning Model (ENSEMBLE).

In [67]:
# Filename: predict_locations_kalman.py

# --- Imports ---
import pandas as pd
import numpy as np
import geopy.distance
import os
import time
import traceback
import joblib
import matplotlib.pyplot as plt # For optional error plotting if evaluating
import warnings
from filterpy.kalman import KalmanFilter
# from filterpy.common import Q_discrete_white_noise # Not used in current KF version

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

# --- Configuration ---
# Data and Paths
PREDICTION_RAW_DATA_FILE = 'dataset/5G_UL.csv' # Or your specific prediction file
RESULTS_DIR = 'result_predict'
MODEL_SAVE_DIR = 'saved_models' # Where trained models are stored
PREDICTION_PREFIX = "UL_Pred_EnsembleKalman_v1" # Unique prefix for this prediction run

# Model and Transformer Filenames (UPDATE THESE from training output)
# These must point to the model and transformer saved by train_model_advanced_ensemble.py
SAVED_MODEL_FILENAME = 'DL_Ensemble_AdvFeat_v1_Ensemble_MOR_Voting_LGBM_RF_KNN_model.joblib'
SAVED_TRANSFORMER_FILENAME = 'DL_Ensemble_AdvFeat_v1_Ensemble_MOR_Voting_LGBM_RF_KNN_columntransformer.joblib'

# Prediction Behavior
EVALUATE_PREDICTIONS = True # Set to True if your prediction data has Latitude/Longitude for error calculation
APPLY_KALMAN_FILTER = True  # Set to False to skip Kalman filtering

# Kalman Filter Parameters (TUNE THESE!)
KF_DT_SECONDS = 1.0               # Default time step if 'Time' column is unusable
KF_R_STD_METERS = 30.0            # Measurement noise std dev (accuracy of ML model)
KF_Q_STD_ACCEL_METERS_SEC2 = 2.0  # Process noise std dev (related to acceleration uncertainty)

# Feature Engineering Config (MUST BE IDENTICAL TO train_model_advanced_ensemble.py)
UNIQUE_PCIS = sorted([3, 13, 23, 30, 40, 48, 59, 68, 76])
FINGERPRINT_COLS_RSRP = [f'RSRP_PCI{pci}' for pci in UNIQUE_PCIS]
FINGERPRINT_COLS_RSRQ = [f'RSRQ_PCI{pci}' for pci in UNIQUE_PCIS]
FINGERPRINT_COLS_OTHER = ['SINR_Serving', 'TA_Serving']
BASE_FINGERPRINT_FEATURES = FINGERPRINT_COLS_RSRP + FINGERPRINT_COLS_RSRQ + FINGERPRINT_COLS_OTHER
NEW_NUMERICAL_RAW_COLS = [
    'merged_row_count', 'NR_UE_Pathloss_DL_0', 'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL',
    'NR_UE_NACK_Rate_DL_0', 'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_MCS_DL_0', 'NR_UE_RB_Num_DL_0',
    'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_BLER_DL_0', 'NR_UE_CCE_AggregationLev_0',
    'NR_UE_Power_Tx_PUSCH_0', 'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0',
    'NR_UE_RACH_Attempt', 'NR_UE_RACH_OK', 'NR_UE_RACH_Fail', 'NR_UE_RACH_Procedure_Count',
    'NR_UE_RRCReEstAttempt', 'NR_UE_RRCReEstFail', 'NR_UE_RRCConnectionAttempt',
    'NR_UE_RRCConnectionSetupOk', 'NR_UE_RRCConnectionComplete', 'NR_UE_RRCConnectionDrop',
    'NR_UE_RRCHOAttempt', 'NR_UE_RRCHOOK'
]
CATEGORICAL_RAW_COLS_CONFIG = {
    'Technology_Mode': {'default': 'Unknown_TechMode'},
    'NR_UE_RRCReEst_EndResult': {'default': 'Unknown_ReEstResult'}
}
ALL_RAW_FEATURE_COL_NAMES = BASE_FINGERPRINT_FEATURES + NEW_NUMERICAL_RAW_COLS + list(CATEGORICAL_RAW_COLS_CONFIG.keys())
LABEL_COLS = ['Latitude', 'Longitude'] # For evaluation

# Default values for fingerprinting and new features (MUST MATCH TRAINING)
NO_SIGNAL_RSRP_DBM = -140.0; NO_SIGNAL_RSRQ_DB = -30.0
DEFAULT_SINR_DB = -20.0; DEFAULT_TA = 0.0
DEFAULT_VALUES_NEW_FEATURES = {
    'merged_row_count': 1.0, 'NR_UE_Pathloss_DL_0': 150.0, 'NR_UE_Throughput_PDCP_DL': 0.0,
    'App_Throughput_DL': 0.0, 'NR_UE_NACK_Rate_DL_0': 0.0, 'NR_UE_Ack_As_Nack_DL_0': 0.0,
    'NR_UE_MCS_DL_0': 0.0, 'NR_UE_RB_Num_DL_0': 0.0, 'NR_UE_Modulation_Avg_DL_0': 0.0,
    'NR_UE_RI_DL_0': 1.0, 'NR_UE_BLER_DL_0': 0.0, 'NR_UE_CCE_AggregationLev_0': 4.0,
    'NR_UE_Power_Tx_PUSCH_0': 0.0, 'NR_UE_Power_Tx_PRACH_0': 0.0, 'NR_UE_NACK_Rate_UL_0': 0.0,
    'NR_UE_RACH_Attempt': 0.0, 'NR_UE_RACH_OK': 0.0, 'NR_UE_RACH_Fail': 0.0,
    'NR_UE_RACH_Procedure_Count': 0.0, 'NR_UE_RRCReEstAttempt': 0.0, 'NR_UE_RRCReEstFail': 0.0,
    'NR_UE_RRCConnectionAttempt': 0.0, 'NR_UE_RRCConnectionSetupOk': 0.0,
    'NR_UE_RRCConnectionComplete': 0.0, 'NR_UE_RRCConnectionDrop': 0.0,
    'NR_UE_RRCHOAttempt': 0.0, 'NR_UE_RRCHOOK': 0.0
}
# Column names in the *prediction* CSV file (may differ from training raw data)
# Based on your `5G_UL.csv` output
SERVING_PCI_COL_PREDICT = 'NR_UE_PCI_0'
SERVING_RSRP_COL_PREDICT = 'NR_UE_RSRP_0'
SERVING_RSRQ_COL_PREDICT = 'NR_UE_RSRQ_0'
SERVING_SINR_COL_PREDICT = 'NR_UE_SINR_0'
SERVING_TA_COL_PREDICT = 'NR_UE_Timing_Advance' # Make sure this exists or handle if not
NEIGHBOR_PCI_COLS_PREDICT_PATTERN = 'NR_UE_Nbr_PCI_{}'
NEIGHBOR_RSRP_COLS_PREDICT_PATTERN = 'NR_UE_Nbr_RSRP_{}'
NEIGHBOR_RSRQ_COLS_PREDICT_PATTERN = 'NR_UE_Nbr_RSRQ_{}'
MAX_NEIGHBORS_PREDICT = 5


# --- File Paths ---
PREDICTION_OUTPUT_FILE = os.path.join(RESULTS_DIR, f'{PREDICTION_PREFIX}_predicted_locations.csv')
PREDICTION_ERROR_CSV_FILE = os.path.join(RESULTS_DIR, f'{PREDICTION_PREFIX}_prediction_errors.csv') # If evaluating
PREDICTION_CDF_PLOT_FILE = PREDICTION_ERROR_CSV_FILE.replace('.csv', '_cdf.png') # If evaluating


os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Functions ---
def process_raw_data_predict(df_raw_predict_input, all_expected_raw_features):
    """Processes raw prediction data to match the feature structure expected by the ColumnTransformer."""
    print("Processing raw prediction data (to match training raw feature structure)...")
    df_processed = df_raw_predict_input.copy()
    
    # 1. Base Fingerprint Features
    for pci in UNIQUE_PCIS: # Global
        df_processed[f'RSRP_PCI{pci}'] = NO_SIGNAL_RSRP_DBM
        df_processed[f'RSRQ_PCI{pci}'] = NO_SIGNAL_RSRQ_DB
    df_processed['SINR_Serving'] = DEFAULT_SINR_DB
    # Ensure TA_Serving column is created even if NR_UE_Timing_Advance isn't in input
    # It's part of ALL_RAW_FEATURE_COL_NAMES
    df_processed['TA_Serving'] = DEFAULT_TA 


    for index, row in df_processed.iterrows():
        s_pci = row.get(SERVING_PCI_COL_PREDICT) # Uses _PREDICT column names
        if pd.notna(s_pci) and int(s_pci) in UNIQUE_PCIS:
            s_pci_int = int(s_pci)
            if pd.notna(row.get(SERVING_RSRP_COL_PREDICT)): df_processed.loc[index, f'RSRP_PCI{s_pci_int}'] = float(row[SERVING_RSRP_COL_PREDICT])
            if pd.notna(row.get(SERVING_RSRQ_COL_PREDICT)): df_processed.loc[index, f'RSRQ_PCI{s_pci_int}'] = float(row[SERVING_RSRQ_COL_PREDICT])
        if pd.notna(row.get(SERVING_SINR_COL_PREDICT)): df_processed.loc[index, 'SINR_Serving'] = float(row[SERVING_SINR_COL_PREDICT])
        # Handle TA_Serving specifically, as it might be missing in some prediction files
        ta_val = row.get(SERVING_TA_COL_PREDICT) # e.g. 'NR_UE_Timing_Advance'
        if pd.notna(ta_val):
            df_processed.loc[index, 'TA_Serving'] = float(ta_val)
        # else: it keeps DEFAULT_TA set above

        for i in range(MAX_NEIGHBORS_PREDICT):
            nbr_pci_val = row.get(NEIGHBOR_PCI_COLS_PREDICT_PATTERN.format(i))
            if pd.notna(nbr_pci_val) and int(nbr_pci_val) in UNIQUE_PCIS:
                nbr_pci_int = int(nbr_pci_val)
                if pd.notna(row.get(NEIGHBOR_RSRP_COLS_PREDICT_PATTERN.format(i))): df_processed.loc[index, f'RSRP_PCI{nbr_pci_int}'] = float(row[NEIGHBOR_RSRP_COLS_PREDICT_PATTERN.format(i)])
                if pd.notna(row.get(NEIGHBOR_RSRQ_COLS_PREDICT_PATTERN.format(i))): df_processed.loc[index, f'RSRQ_PCI{nbr_pci_int}'] = float(row[NEIGHBOR_RSRQ_COLS_PREDICT_PATTERN.format(i)])
    
    # 2. New Numerical Features
    for col in NEW_NUMERICAL_RAW_COLS: # Global
        if col in df_processed.columns: # Check if column exists in input CSV
            df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
            df_processed[col].fillna(DEFAULT_VALUES_NEW_FEATURES[col], inplace=True)
        else: # If not in input CSV, create it with default (as ColumnTransformer expects it)
            df_processed[col] = DEFAULT_VALUES_NEW_FEATURES[col]
            
    # 3. Categorical Features
    for col, params in CATEGORICAL_RAW_COLS_CONFIG.items(): # Global
        if col in df_processed.columns: # Check if column exists in input CSV
            df_processed[col].fillna(params['default'], inplace=True)
        else: # If not in input CSV, create it with default
            df_processed[col] = params['default']

    # Ensure all expected columns for the transformer are present, in the correct order
    # This uses the globally defined ALL_RAW_FEATURE_COL_NAMES from training config
    missing_cols = [col for col in all_expected_raw_features if col not in df_processed.columns]
    if missing_cols:
        print(f"FATAL: Raw feature columns expected by ColumnTransformer are missing from processed prediction data: {missing_cols}")
        print(f"       Available columns: {df_processed.columns.tolist()}")
        print(f"       Expected columns based on training: {all_expected_raw_features}")
        return None
        
    return df_processed[all_expected_raw_features] # Return only the required features in order

def apply_kalman_filter(predictions_df, r_std_m, q_std_accel, dt_s):
    print(f"Applying Kalman Filter (R_std={r_std_m}m, Q_std_accel={q_std_accel}m/s^2, dt={dt_s}s)...")
    filtered_lat, filtered_lon = [], []
    
    for coord_type, measurements, r_val, q_val in [
        ('lat', predictions_df['predicted_latitude'], r_std_m, q_std_accel),
        ('lon', predictions_df['predicted_longitude'], r_std_m, q_std_accel)
    ]:
        kf = KalmanFilter(dim_x=2, dim_z=1)
        kf.F = np.array([[1., dt_s], [0., 1.]])
        kf.H = np.array([[1., 0.]])
        kf.R = np.array([[r_val**2]])
        
        # Process noise: Q_discrete_white_noise expects variance of the highest derivative
        # If q_val is std dev of acceleration, then var = q_val**2
        # For a 2-state (pos, vel) system where noise is in acceleration:
        G = np.array([[0.5*dt_s**2], [dt_s]]) # Noise gain matrix for acceleration input
        kf.Q = G @ G.T * (q_val**2)
        # Alternative simpler Q definition (less physically rigorous but common):
        # kf.Q = np.diag([ (0.5*dt_s**2*q_val)**2 , (dt_s*q_val)**2 ])


        if measurements.empty or pd.isna(measurements.iloc[0]):
            print(f"Warning: First measurement for {coord_type} is NaN or series is empty. Cannot initialize KF well.")
            # Fill with NaNs or some other strategy if this happens often
            if coord_type == 'lat': filtered_lat = [np.nan] * len(measurements)
            else: filtered_lon = [np.nan] * len(measurements)
            continue # Skip this coordinate if no valid start

        kf.x = np.array([measurements.iloc[0], 0.]) # Initial state: [pos, vel=0]
        kf.P = np.diag([r_val**2, (q_val*dt_s)**2 if dt_s > 0 else q_val**2]) # Initial uncertainty

        current_filtered_coords = []
        last_valid_kf_x0 = measurements.iloc[0] # Fallback for NaN measurements

        for z_val in measurements:
            if pd.isna(z_val):
                kf.predict() # Predict forward if measurement is missing
                current_filtered_coords.append(last_valid_kf_x0) # Use last known good filtered value
                # Or append kf.x[0] if you prefer the pure prediction
            else:
                kf.predict()
                kf.update(z_val)
                current_filtered_coords.append(kf.x[0])
                last_valid_kf_x0 = kf.x[0]
            
        if coord_type == 'lat': filtered_lat = current_filtered_coords
        else: filtered_lon = current_filtered_coords
            
    return pd.DataFrame({'kf_latitude': filtered_lat, 'kf_longitude': filtered_lon})

def calculate_errors_for_eval(y_true_df, y_pred_array, model_name_print="Model"):
    # Simplified version of calculate_errors from training script for just getting stats
    errors = []
    if not isinstance(y_true_df, pd.DataFrame) or y_true_df.empty or \
       not isinstance(y_pred_array, np.ndarray) or y_pred_array.shape[1] != 2 or \
       len(y_true_df) != len(y_pred_array):
        print(f"({model_name_print}) Invalid input for error calculation. True: {type(y_true_df)}, Pred: {type(y_pred_array)}")
        return np.array([]), pd.DataFrame() # Return empty if inputs are bad

    y_true_reset = y_true_df.reset_index(drop=True)
    error_details_list = []

    for i in range(len(y_true_reset)):
        true_lat, true_lon = y_true_reset.loc[i, 'Latitude'], y_true_reset.loc[i, 'Longitude']
        pred_lat, pred_lon = y_pred_array[i, 0], y_pred_array[i, 1]
        
        detail = {'true_lat': true_lat, 'true_lon': true_lon, 'pred_lat': pred_lat, 'pred_lon': pred_lon, 'error_m': np.nan}

        if not all(pd.notna(v) for v in [true_lat, true_lon, pred_lat, pred_lon]):
            error_details_list.append(detail); continue
        if not (-90 <= pred_lat <= 90 and -180 <= pred_lon <= 180):
            error_details_list.append(detail); continue
        try:
            distance = geopy.distance.geodesic((true_lat, true_lon), (pred_lat, pred_lon)).meters
            errors.append(distance)
            detail['error_m'] = distance
        except ValueError:
            pass # Error remains NaN
        error_details_list.append(detail)
        
    return np.array(errors), pd.DataFrame(error_details_list)


# --- Main Prediction Script ---
if __name__ == '__main__':
    print(f"--- Starting Prediction Script: {PREDICTION_PREFIX} ---")
    script_start_time = time.time()

    # 1. Load Model and Transformer
    print(f"\n[Step 1/6] Loading Model and ColumnTransformer...")
    model_full_path = os.path.join(MODEL_SAVE_DIR, SAVED_MODEL_FILENAME)
    transformer_full_path = os.path.join(MODEL_SAVE_DIR, SAVED_TRANSFORMER_FILENAME)
    loaded_ml_model, loaded_col_transformer = None, None
    try:
        if not os.path.exists(model_full_path): raise FileNotFoundError(f"Model file missing: {model_full_path}")
        loaded_ml_model = joblib.load(model_full_path)
        print(f"Model loaded from: {model_full_path}")
        if not os.path.exists(transformer_full_path): raise FileNotFoundError(f"Transformer file missing: {transformer_full_path}")
        loaded_col_transformer = joblib.load(transformer_full_path)
        print(f"ColumnTransformer loaded from: {transformer_full_path}")
    except Exception as e: print(f"FATAL: Error loading model/transformer: {e}"); traceback.print_exc(); sys.exit()

    # 2. Load and Process Raw Prediction Data
    print(f"\n[Step 2/6] Loading and Processing Raw Prediction Data...")
    raw_features_for_predict, original_df_predict_full = None, None
    try:
        print(f"Loading raw prediction data from {PREDICTION_RAW_DATA_FILE}...")
        try: # Robust CSV reading
            original_df_predict_full = pd.read_csv(PREDICTION_RAW_DATA_FILE, low_memory=False, delimiter=';')
            if SERVING_PCI_COL_PREDICT not in original_df_predict_full.columns: # Check essential col
                 print(f"Warning: Column '{SERVING_PCI_COL_PREDICT}' not found with ';' delimiter. Trying default ','.")
                 original_df_predict_full = pd.read_csv(PREDICTION_RAW_DATA_FILE, low_memory=False)
                 if SERVING_PCI_COL_PREDICT not in original_df_predict_full.columns:
                     raise ValueError(f"Essential column '{SERVING_PCI_COL_PREDICT}' not found with common delimiters.")
            print(f"Successfully loaded {PREDICTION_RAW_DATA_FILE}. Rows: {len(original_df_predict_full)}")
        except Exception as read_err: print(f"Pandas read_csv FAILED: {read_err}"); traceback.print_exc(); sys.exit()

        if original_df_predict_full.empty: print("Prediction input file is empty. Exiting."); sys.exit()
        
        # ALL_RAW_FEATURE_COL_NAMES is defined globally from training config
        raw_features_for_predict = process_raw_data_predict(original_df_predict_full, ALL_RAW_FEATURE_COL_NAMES)
        if raw_features_for_predict is None or raw_features_for_predict.empty:
            print("FATAL: Raw feature processing for prediction failed."); sys.exit()
        print(f"Successfully processed into {raw_features_for_predict.shape[1]} raw features for {len(raw_features_for_predict)} samples.")
    except FileNotFoundError: print(f"FATAL: Prediction file not found: {PREDICTION_RAW_DATA_FILE}"); sys.exit()
    except Exception as e: print(f"FATAL: Error during prediction data load/process: {e}"); traceback.print_exc(); sys.exit()

    # 3. Apply Loaded ColumnTransformer
    print(f"\n[Step 3/6] Applying Loaded ColumnTransformer...")
    transformed_features_predict = None
    try:
        if loaded_col_transformer is None: raise ValueError("ColumnTransformer is None.")
        transformed_features_predict = loaded_col_transformer.transform(raw_features_for_predict)
        print(f"ColumnTransformer applied. Transformed features shape: {transformed_features_predict.shape}")
    except Exception as e: print(f"FATAL: Error applying ColumnTransformer: {e}"); traceback.print_exc(); sys.exit()

    # 4. Make Initial ML Predictions
    print(f"\n[Step 4/6] Making Initial ML Predictions...")
    initial_ml_predictions_array = None
    pred_start_time = time.time()
    try:
        initial_ml_predictions_array = loaded_ml_model.predict(transformed_features_predict)
        if initial_ml_predictions_array.shape[1] != 2 or len(initial_ml_predictions_array) != len(transformed_features_predict):
             raise ValueError("Initial prediction output shape mismatch.")
    except Exception as e: print(f"FATAL: Error during initial prediction: {e}"); traceback.print_exc(); sys.exit()
    print(f"Initial ML prediction finished in {time.time() - pred_start_time:.2f}s.")
    
    # Store initial predictions with original index
    initial_predictions_df = pd.DataFrame(
        initial_ml_predictions_array, 
        columns=['ml_latitude', 'ml_longitude'], 
        index=original_df_predict_full.index 
    )

    # 5. Apply Kalman Filter (Optional)
    kalman_filtered_predictions_df = None
    if APPLY_KALMAN_FILTER:
        print(f"\n[Step 5/6] Applying Kalman Filter Post-Processing...")
        try:
            dt_actual = KF_DT_SECONDS # Use configured default
            if 'Time' in original_df_predict_full.columns:
                try:
                    time_col_s = pd.to_datetime(original_df_predict_full['Time'], errors='coerce')
                    if time_col_s.notna().sum() > 1: # Need at least 2 valid times
                        time_diffs_s = time_col_s.diff().dt.total_seconds()
                        # Use median of valid diffs, or fill with configured default
                        valid_time_diffs = time_diffs_s[time_diffs_s > 0] # Only positive diffs
                        if not valid_time_diffs.empty:
                            dt_actual = valid_time_diffs.median() # Median is robust to outliers
                            print(f"  Estimated median dt from 'Time' column: {dt_actual:.3f}s (using this for KF)")
                        else:
                            print(f"  No valid positive time differences. Using configured dt={KF_DT_SECONDS}s.")
                    else:
                         print(f"  Not enough valid time data. Using configured dt={KF_DT_SECONDS}s.")
                except Exception as time_err:
                    print(f"  Warning: Could not parse 'Time' for dt: {time_err}. Using configured dt={KF_DT_SECONDS}s.")
            else:
                print(f"  'Time' column not found. Using configured dt={KF_DT_SECONDS}s.")
            
            # Temporary rename for apply_kalman_filter function
            temp_kf_input_df = initial_predictions_df.rename(columns={'ml_latitude':'predicted_latitude', 'ml_longitude':'predicted_longitude'})

            kalman_filtered_predictions_df = apply_kalman_filter(
                temp_kf_input_df,
                r_std_m=KF_R_STD_METERS,
                q_std_accel=KF_Q_STD_ACCEL_METERS_SEC2,
                dt_s=dt_actual
            )
            kalman_filtered_predictions_df.index = original_df_predict_full.index # Ensure index alignment
            print("Kalman filtering application complete.")
        except Exception as kf_err:
            print(f"Error during Kalman Filter application: {kf_err}"); traceback.print_exc()
            kalman_filtered_predictions_df = None # Fallback
    else:
        print(f"\n[Step 5/6] Skipping Kalman Filter as per configuration.")

    # 6. Format and Save Final Predictions
    print(f"\n[Step 6/6] Formatting and Saving Final Predictions...")
    final_output_df = original_df_predict_full.copy()
    final_output_df = final_output_df.merge(initial_predictions_df, left_index=True, right_index=True, how='left')

    if kalman_filtered_predictions_df is not None and not kalman_filtered_predictions_df.empty:
        final_output_df = final_output_df.merge(kalman_filtered_predictions_df, left_index=True, right_index=True, how='left')
        # Use KF as primary if available, keep ML as backup/comparison
        final_output_df.rename(columns={'kf_latitude': 'predicted_latitude', 
                                        'kf_longitude': 'predicted_longitude'}, inplace=True)
        print("Using Kalman Filtered predictions as 'predicted_latitude/longitude'. Original ML are 'ml_latitude/longitude'.")
    else: # KF not applied or failed, use initial ML as primary
        final_output_df.rename(columns={'ml_latitude': 'predicted_latitude', 
                                        'ml_longitude': 'predicted_longitude'}, inplace=True)
        print("Using initial ML predictions as 'predicted_latitude/longitude'.")

    # Reorder columns for clarity
    final_pred_cols = []
    if 'predicted_latitude' in final_output_df.columns: final_pred_cols.extend(['predicted_latitude', 'predicted_longitude'])
    if 'ml_latitude' in final_output_df.columns and 'ml_latitude' not in final_pred_cols: final_pred_cols.extend(['ml_latitude', 'ml_longitude'])
    
    other_output_cols = [col for col in final_output_df.columns if col not in final_pred_cols]
    final_output_df = final_output_df[other_output_cols + final_pred_cols]

    try:
        final_output_df.to_csv(PREDICTION_OUTPUT_FILE, index=False, float_format='%.7f')
        print(f"Final predictions saved to: {PREDICTION_OUTPUT_FILE}")
    except Exception as save_err: print(f"Error saving final predictions: {save_err}"); traceback.print_exc()

    # Optional: Evaluate Predictions
    if EVALUATE_PREDICTIONS and all(col in original_df_predict_full.columns for col in LABEL_COLS):
        print("\n--- Evaluating Predictions ---")
        y_true_for_eval = original_df_predict_full.loc[final_output_df.index, LABEL_COLS].copy()

        if 'ml_latitude' in final_output_df.columns:
            y_pred_ml_eval_arr = final_output_df[['ml_latitude', 'ml_longitude']].values
            errors_ml, details_ml_df = calculate_errors_for_eval(y_true_for_eval, y_pred_ml_eval_arr, "Initial_ML")
            if len(errors_ml) > 0:
                print(f"  Initial ML - Median Error: {np.median(errors_ml):.2f}m, 95th Pctl: {np.percentile(errors_ml, 95):.2f}m")
                # Optionally save details_ml_df or plot CDF

        # Evaluate final output (which might be KF or ML if KF failed/skipped)
        if 'predicted_latitude' in final_output_df.columns:
            y_pred_final_eval_arr = final_output_df[['predicted_latitude', 'predicted_longitude']].values
            errors_final, details_final_df = calculate_errors_for_eval(y_true_for_eval, y_pred_final_eval_arr, "Final_Output")
            if len(errors_final) > 0:
                print(f"  Final Output - Median Error: {np.median(errors_final):.2f}m, 95th Pctl: {np.percentile(errors_final, 95):.2f}m")
                if not details_final_df.empty:
                    details_final_df.to_csv(PREDICTION_ERROR_CSV_FILE, index=False, float_format='%.6f')
                    print(f"  Final output error details saved to: {PREDICTION_ERROR_CSV_FILE}")
                try: # Plot CDF for final output
                    plt.figure(figsize=(8,6)); s_errs_final=np.sort(errors_final); cdf_vals_final=np.arange(1,len(s_errs_final)+1)/len(s_errs_final)
                    plt.plot(s_errs_final, cdf_vals_final*100, marker='.', linestyle='-', markersize=2)
                    plt.xlabel("Positioning Error (m)"); plt.ylabel("CDF (%)"); plt.title(f"Final Prediction Error CDF ({PREDICTION_PREFIX})")
                    plt.grid(True,which='both',linestyle='--',linewidth=0.5); plt.ylim(0,100); plt.xlim(left=0)
                    plt.axvline(np.median(errors_final),color='orange',linestyle=':',lw=1,label=f'Median ({np.median(errors_final):.1f}m)')
                    p95_final=np.percentile(errors_final,95); plt.axvline(p95_final,color='red',linestyle=':',lw=1,label=f'95th ({p95_final:.1f}m)')
                    plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(PREDICTION_CDF_PLOT_FILE); print(f"  Final output CDF plot saved to: {PREDICTION_CDF_PLOT_FILE}"); plt.close()
                except Exception as plot_err: print(f"  Error plotting final CDF: {plot_err}")
    else:
        print("\nSkipping evaluation: EVALUATE_PREDICTIONS is False or label columns missing in prediction input.")

    script_total_time = time.time() - script_start_time
    print(f"\n--- Prediction Script Finished in {script_total_time:.2f} seconds ({script_total_time/60:.1f} minutes) ---")
    if APPLY_KALMAN_FILTER:
        print("\nKalman Filter Tuning Notes:")
        print(f"- R_std: {KF_R_STD_METERS}m (ML model error), Q_std_accel: {KF_Q_STD_ACCEL_METERS_SEC2}m/s^2 (motion uncertainty), dt: {dt_actual if 'dt_actual' in locals() else KF_DT_SECONDS}s (time step).")
        print("- TUNE THESE parameters based on your data characteristics for optimal KF performance.")

--- Starting Prediction Script: UL_Pred_EnsembleKalman_v1 ---

[Step 1/6] Loading Model and ColumnTransformer...
Model loaded from: saved_models\DL_Ensemble_AdvFeat_v1_Ensemble_MOR_Voting_LGBM_RF_KNN_model.joblib
ColumnTransformer loaded from: saved_models\DL_Ensemble_AdvFeat_v1_Ensemble_MOR_Voting_LGBM_RF_KNN_columntransformer.joblib

[Step 2/6] Loading and Processing Raw Prediction Data...
Loading raw prediction data from dataset/5G_UL.csv...
Successfully loaded dataset/5G_UL.csv. Rows: 49905
Processing raw prediction data (to match training raw feature structure)...
Successfully processed into 49 raw features for 49905 samples.

[Step 3/6] Applying Loaded ColumnTransformer...
ColumnTransformer applied. Transformed features shape: (49905, 50)

[Step 4/6] Making Initial ML Predictions...
Initial ML prediction finished in 1.99s.

[Step 5/6] Applying Kalman Filter Post-Processing...
  No valid positive time differences. Using configured dt=1.0s.
Applying Kalman Filter (R_std=30.0m, Q_st